# setup and imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import textwrap
import seaborn as sns
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib for inline display
%matplotlib inline
plt.style.use('default')

print("✅ Imports Complete")

# 4x4 human evals plot

In [ ]:
def load_clean_and_sort_data(filepath):
    """Load, clean and sort human evaluation data"""
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"FATAL ERROR: The file '{filepath}' was not found.")
        return None

    df['Code Language'] = df['Code Language'].str.lower()
    df['Prompt ID'] = pd.to_numeric(df['Prompt ID'], errors='coerce')
    df.dropna(subset=['Prompt ID'], inplace=True)
    df['Prompt ID'] = df['Prompt ID'].astype(int)

    count_columns = [col for col in df.columns if '(count of 5s)' in col]
    for col in count_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    df['Topic'] = df['Topic'].str.replace('‑', ' ', regex=False)
    df.sort_values(by='Prompt ID', inplace=True)
    return df

def create_and_save_canvas_plot(detailed_df, overall_df, metric_name, filename):
    """Generate 4x4 canvas plot"""
    fig, axs = plt.subplots(4, 4, figsize=(24, 22))
    fig.suptitle(f'Human Evaluations - Metric: {metric_name}', fontsize=28, y=0.98)

    colors = {'OpenAI o4-mini-high': '#FF8080', 'Gemini 2.5 pro': '#80FF80', 'Claude Opus 4': '#8080FF'}
    model_legend_names = {'g4': 'OpenAI o4-mini-high', 'gemini': 'Gemini 2.5 pro', 'claude': 'Claude Opus 4'}
    language_order = ['C++', 'Python', 'Java', 'JavaScript']
    lang_code_map = {'C++': 'cpp', 'Python': 'python', 'Java': 'java', 'JavaScript': 'javascript'}

    language_label_colors = {
        'C++': '#00FFFF',
        'Python': '#3776AB',
        'Java': '#ed8b00',
        'JavaScript': '#f7df1e'
    }

    g4_col = f'o4-mini-high - {metric_name} (count of 5s)'
    gemini_col = f'Gemini 2.5 Pro - {metric_name} (count of 5s)'
    claude_col = f'Claude Opus 4 - {metric_name} (count of 5s)'

    for row_idx, lang_name in enumerate(language_order):
        lang_code = lang_code_map[lang_name]
        lang_overall_data = overall_df[overall_df['Code Language'] == lang_code]
        lang_detailed_data = detailed_df[detailed_df['Code Language'] == lang_code]

        row_max_y = 0
        if not lang_overall_data.empty:
            row_max_y = max(row_max_y, lang_overall_data[[g4_col, gemini_col, claude_col]].max().max())
        if not lang_detailed_data.empty:
            row_max_y = max(row_max_y, lang_detailed_data[[g4_col, gemini_col, claude_col]].max().max())
        common_y_limit = max(row_max_y + 2, 5)

        # Overall Chart (Column 0)
        ax_overall = axs[row_idx, 0]
        if not lang_overall_data.empty:
            industries = lang_overall_data['Industry'].tolist()
            x = np.arange(len(industries))
            width = 0.25

            ax_overall.bar(x - width, lang_overall_data[g4_col], width, color=colors[model_legend_names['g4']])
            ax_overall.bar(x, lang_overall_data[gemini_col], width, color=colors[model_legend_names['gemini']])
            ax_overall.bar(x + width, lang_overall_data[claude_col], width, color=colors[model_legend_names['claude']])

            lang_color = language_label_colors[lang_name]
            ax_overall.set_title('Overall', y=1.05, fontweight='bold', color='black', fontsize=16,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=lang_color, alpha=0.7))

            wrapped_labels = [textwrap.fill(l, 12) for l in industries]
            ax_overall.set_xticks(x)
            ax_overall.set_xticklabels(wrapped_labels, fontsize=10, ha='center')
            ax_overall.set_ylim(0, common_y_limit)
            ax_overall.set_yticks(np.arange(0, common_y_limit, step=max(1, int(common_y_limit/5))))
            ax_overall.grid(axis='y', linestyle='--', alpha=0.7)

        # Detailed Industry Charts (Columns 1, 2, 3)
        industries_in_lang = lang_overall_data['Industry'].tolist()

        for col_offset, industry_name in enumerate(industries_in_lang[:3]):
            ax = axs[row_idx, col_offset + 1]
            industry_slice = lang_detailed_data[lang_detailed_data['Industry'] == industry_name]

            subtopics = industry_slice['Topic'].tolist()
            x = np.arange(len(subtopics))
            width = 0.25

            ax.bar(x - width, industry_slice[g4_col], width, color=colors[model_legend_names['g4']])
            ax.bar(x, industry_slice[gemini_col], width, color=colors[model_legend_names['gemini']])
            ax.bar(x + width, industry_slice[claude_col], width, color=colors[model_legend_names['claude']])

            lang_color = language_label_colors[lang_name]
            ax.set_title(industry_name, y=1.05, fontweight='bold', color='black', fontsize=16,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=lang_color, alpha=0.7))

            wrapped_subtopics = [textwrap.fill(s, 15) for s in subtopics]
            ax.set_xticks(x)
            ax.set_xticklabels(wrapped_subtopics, rotation=0, ha='center', fontsize=8)
            ax.set_ylim(0, common_y_limit)
            ax.set_yticks(np.arange(0, common_y_limit, step=max(1, int(common_y_limit/5))))
            ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Add axis labels
    for row_idx, lang_name in enumerate(language_order):
        axs[row_idx, 0].set_xlabel('Industry', fontsize=12, fontweight='bold')
        for col_idx in range(1, 4):
            if axs[row_idx, col_idx].has_data():
                axs[row_idx, col_idx].set_xlabel('Topic', fontsize=10, fontweight='bold')

    # Hide empty subplots
    for ax in axs.flat:
        if not ax.has_data():
            ax.set_visible(False)

    plt.tight_layout(rect=[0.05, 0.08, 1, 0.96], h_pad=3.0)

    # Language labels
    for row_idx, lang_name in enumerate(language_order):
        pos = axs[row_idx, 0].get_position()
        y_fig_coord = pos.y0 + pos.height / 2
        x_fig_coord = 0.04
        lang_box_color = language_label_colors[lang_name]
        fig.text(x_fig_coord, y_fig_coord, lang_name, ha='center', va='center', rotation=90,
                fontsize=18, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3", fc=lang_box_color, ec="black", lw=1, alpha=0.5))

    # Legend and chart guide
    handles = [plt.Rectangle((0,0),1,1, color=colors[label]) for label in model_legend_names.values()]
    shared_y_coord = 0.04

    legend = fig.legend(handles, model_legend_names.values(),
                    loc='center', bbox_to_anchor=(0.35, shared_y_coord),
                    ncol=3, title='Models', title_fontsize=14, fontsize=12,
                    frameon=True, fancybox=True, shadow=True,
                    edgecolor='gray', framealpha=1, facecolor='white')
    legend.get_frame().set_linewidth(1.5)

    guide_text_raw = "Chart Guide\nX-Axis: Count of '5' Ratings"
    fig.text(0.65, shared_y_coord, guide_text_raw, ha='center', va='center',
            fontsize=12, linespacing=1.5,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                    edgecolor='gray', lw=1.5))

    plt.savefig(filename, dpi=300)
    plt.show()  # Display in notebook
    plt.close(fig)
    print(f"✅ Canvas saved: {filename}")

# EXECUTE: Generate 4x4 Human Evaluation Canvas Plots
print("🎨 Generating 4x4 Human Evaluation Canvas Plots...")

csv_filepath = 'Untitled spreadsheet - Sheet1 (4).csv'  # Update this path
df = load_clean_and_sort_data(csv_filepath)

if df is not None:
    output_dir = "4x4humanevals"
    os.makedirs(output_dir, exist_ok=True)
    print(f"All canvas graphs will be saved in: '{output_dir}'")

    count_columns = [col for col in df.columns if '(count of 5s)' in col]
    agg_operations = {col: 'sum' for col in count_columns}
    agg_operations['Prompt ID'] = 'min'

    overall_agg_df = df.groupby(['Code Language', 'Industry']).agg(agg_operations).reset_index()
    overall_agg_df.sort_values(by='Prompt ID', inplace=True)

    metrics_to_plot = ['Completeness', 'Correctness', 'Relevance']

    for metric in metrics_to_plot:
        print(f"\n--- Generating canvas for Metric: '{metric}' ---")
        filename = f"Canvas_{metric}.png"
        full_path = os.path.join(output_dir, filename)
        create_and_save_canvas_plot(df, overall_agg_df, metric, full_path)
else:
    print("❌ Could not load data. Check file path.")

# indv. human evals - Language - Overall and Industry Wise

In [ ]:
def create_industry_plot(language, industry, data_slice, metric_name, filename):
    """Create individual industry plot"""
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = {'o4-mini-high': '#FF8080', 'Gemini 2.5 pro': '#80FF80', 'Claude Opus 4': '#8080FF'}
    model_names_in_legend = {'g4': 'o4-mini-high', 'gemini': 'Gemini 2.5 pro', 'claude': 'Claude Opus 4'}

    subtopics = data_slice['Topic'].tolist()
    x = np.arange(len(subtopics))
    width = 0.25

    g4_col = f'o4-mini-high - {metric_name} (count of 5s)'
    gemini_col = f'Gemini 2.5 Pro - {metric_name} (count of 5s)'
    claude_col = f'Claude Opus 4 - {metric_name} (count of 5s)'

    g4_counts = data_slice[g4_col].tolist()
    gemini_counts = data_slice[gemini_col].tolist()
    claude_counts = data_slice[claude_col].tolist()

    ax.bar(x - width, g4_counts, width, label=model_names_in_legend['g4'], color=colors[model_names_in_legend['g4']])
    ax.bar(x, gemini_counts, width, label=model_names_in_legend['gemini'], color=colors[model_names_in_legend['gemini']])
    ax.bar(x + width, claude_counts, width, label=model_names_in_legend['claude'], color=colors[model_names_in_legend['claude']])

    ax.set_ylabel("Count of '5' Ratings", fontsize=12, fontweight='bold')
    ax.set_xlabel("Topics", fontsize=12, fontweight='bold')
    ax.set_title(f'{language} - {industry}', fontsize=16, fontweight='bold')

    max_count = max(max(g4_counts, default=0), max(gemini_counts, default=0), max(claude_counts, default=0))
    y_limit = max(max_count + 1, 3)
    ax.set_ylim(0, y_limit)
    ax.set_yticks(np.arange(0, y_limit + 1, 1))

    wrapped_subtopics = [textwrap.fill(s, 20) for s in subtopics]
    ax.set_xticks(x)
    ax.set_xticklabels(wrapped_subtopics, fontsize=9)
    ax.tick_params(axis='x', pad=10)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.legend(title='Models', fontsize=11)

    fig.suptitle(f'Evaluation Metric: {metric_name}', fontsize=14, color='black')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(filename, dpi=150)
    plt.show()  # Display in notebook
    plt.close(fig)
    print(f"✅ Saved: {filename}")

def create_overall_plot(language, overall_data_slice, metric_name, filename):
    """Create overall language plot"""
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = {'o4-mini-high': '#FF8080', 'Gemini 2.5 pro': '#80FF80', 'Claude Opus 4': '#8080FF'}
    model_names_in_legend = {'g4': 'o4-mini-high', 'gemini': 'Gemini 2.5 pro', 'claude': 'Claude Opus 4'}

    industries = overall_data_slice['Industry'].tolist()
    x = np.arange(len(industries))
    width = 0.25

    g4_col = f'o4-mini-high - {metric_name} (count of 5s)' #
    gemini_col = f'Gemini 2.5 Pro - {metric_name} (count of 5s)'
    claude_col = f'Claude Opus 4 - {metric_name} (count of 5s)'

    g4_counts = overall_data_slice[g4_col].tolist()
    gemini_counts = overall_data_slice[gemini_col].tolist()
    claude_counts = overall_data_slice[claude_col].tolist()

    ax.bar(x - width, g4_counts, width, label=model_names_in_legend['g4'], color=colors[model_names_in_legend['g4']])
    ax.bar(x, gemini_counts, width, label=model_names_in_legend['gemini'], color=colors[model_names_in_legend['gemini']])
    ax.bar(x + width, claude_counts, width, label=model_names_in_legend['claude'], color=colors[model_names_in_legend['claude']])

    ax.set_ylabel("Total Count of '5' Ratings (All Topics)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Industries", fontsize=12, fontweight='bold')
    ax.set_title(f'{language} - Overall', fontsize=16, fontweight='bold')

    max_count = max(max(g4_counts, default=0), max(gemini_counts, default=0), max(claude_counts, default=0))
    y_limit = max(max_count + 1, 5)
    ax.set_ylim(0, y_limit)
    ax.set_yticks(np.arange(0, y_limit + 1, step=max(1, int(y_limit/5))))

    wrapped_industries = [textwrap.fill(s, 18) for s in industries]
    ax.set_xticks(x)
    ax.set_xticklabels(wrapped_industries, fontsize=10)
    ax.tick_params(axis='x', pad=10)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.legend(title='Models', fontsize=11)

    fig.suptitle(f'Evaluation Metric: {metric_name}', fontsize=14, color='black')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(filename, dpi=150)
    plt.show()  # Display in notebook
    plt.close(fig)
    print(f"✅ Saved: {filename}")

# EXECUTE: Generate Individual Human Evaluation Plots
print("📈 Generating Individual Human Evaluation Plots...")

csv_filepath = 'Untitled spreadsheet - Sheet1 (4).csv'  # Update this path
df = load_clean_and_sort_data(csv_filepath)

if df is not None:
    output_dir_industry = "Individual_human_evals"
    output_dir_overall = "humanevals_overall"
    os.makedirs(output_dir_industry, exist_ok=True)
    os.makedirs(output_dir_overall, exist_ok=True)
    print(f"Detailed graphs will be saved in: '{output_dir_industry}'")
    print(f"Overall graphs will be saved in: '{output_dir_overall}'")

    count_columns = [col for col in df.columns if '(count of 5s)' in col]
    agg_logic = {col: 'sum' for col in count_columns}
    agg_logic['Prompt ID'] = 'min'
    overall_agg_df = df.groupby(['Code Language', 'Industry']).agg(agg_logic).reset_index()
    overall_agg_df.sort_values(by='Prompt ID', inplace=True)

    metrics_to_plot = ['Completeness', 'Correctness', 'Relevance']

    for metric in metrics_to_plot:
        print(f"\n--- Generating graphs for Metric: '{metric}' ---")

        # Generate Industry-Specific Graphs
        print(" Generating detailed industry-topic graphs...")
        industry_grouped = df.groupby(['Code Language', 'Industry'], sort=False)

        for (lang, industry), group_df in industry_grouped:
            display_lang = 'C++' if lang.lower() == 'cpp' else lang.capitalize()
            filename = f"{metric}_{display_lang.replace('C++', 'Cpp')}-{industry.replace(' ', '_')}.png"
            full_path = os.path.join(output_dir_industry, filename)
            create_industry_plot(display_lang, industry, group_df, metric, full_path)

        # Generate Overall Language Graphs
        print(" Generating overall language graphs...")
        overall_grouped = overall_agg_df.groupby('Code Language', sort=False)

        for lang, lang_group_df in overall_grouped:
            display_lang = 'C++' if lang.lower() == 'cpp' else lang.capitalize()
            filename = f"{metric}_{display_lang.replace('C++', 'Cpp')}-Overall.png"
            full_path = os.path.join(output_dir_overall, filename)
            create_overall_plot(display_lang, lang_group_df, metric, full_path)
else:
    print("❌ Could not load data. Check file path.")

# 4x4 system evals

In [ ]:
# =============================================================================
# CELL 4: 4x4 SYSTEM EVALUATIONS CROSS-LANGUAGE PLOT
# =============================================================================

def get_overall_data_for_language(df, language):
    """Extract overall industry data for a specific language"""
    lang_data = df[df['Code Language'] == language].copy()
    language_industries = {
        'C++': ['Gaming', 'IoT', 'Security'],
        'Python': ['Machine Learning', 'FinTech', 'EdTech'],
        'Java': ['E‑Commerce', 'CRM', 'Hotel'],
        'JavaScript': ['Social Networking', 'Media', 'Streaming']
    }

    industries = language_industries.get(language, [])
    score_cols = [
        'Chat GPT o4 mini-high Codebleu Score',
        'Gemini 2.5 pro Codebleu Score',
        'Claude Opus 4 Codebleu Score'
    ]

    industry_data = []
    industry_labels = []

    for industry in industries:
        industry_rows = lang_data[lang_data['Domain'] == industry]
        if len(industry_rows) > 0:
            industry_scores = {}
            for col in score_cols:
                if col in industry_rows.columns:
                    scores = industry_rows[col].dropna()
                    industry_scores[col] = scores.mean() if len(scores) > 0 else 0.0
                else:
                    industry_scores[col] = 0.0

            industry_data.append(industry_scores)
            industry_labels.append(industry)

    return industry_data, industry_labels

def create_enhanced_cross_language_domain_plot(csv_file):
    """Create comprehensive cross-language domain comparison plot"""
    if not os.path.exists(csv_file):
        print(f"❌ Error: File '{csv_file}' not found!")
        return None

    try:
        df = pd.read_csv(csv_file)
        print(f"✅ Successfully loaded {csv_file}")
        print(f"📊 Dataset shape: {df.shape}")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return None

    languages = ['C++', 'Python', 'Java', 'JavaScript']
    language_domains = {
        'C++': ['Gaming', 'IoT', 'Security'],
        'Python': ['Machine Learning', 'FinTech', 'EdTech'],
        'Java': ['E‑Commerce', 'CRM', 'Hotel'],
        'JavaScript': ['Social Networking', 'Media', 'Streaming']
    }

    language_colors = {
        "C++": "#00FFFF", "Python": "#3776AB",
        "JavaScript": "#F7DF1E", "Java": "#ED8B00"
    }

    score_cols = [
        'Chat GPT o4 mini-high Codebleu Score',
        'Gemini 2.5 pro Codebleu Score',
        'Claude Opus 4 Codebleu Score'
    ]

    model_display_names = {
        'Chat GPT o4 mini-high Codebleu Score': 'o4-mini-high',
        'Gemini 2.5 pro Codebleu Score': 'Gemini 2.5 Pro',
        'Claude Opus 4 Codebleu Score': 'Claude Opus 4'
    }

    model_colors = {
        'Chat GPT o4 mini-high Codebleu Score': '#FF8080',
        'Gemini 2.5 pro Codebleu Score': '#80FF80',
        'Claude Opus 4 Codebleu Score': '#8080FF'
    }

    max_domains = max(len(domains) for domains in language_domains.values())
    n_rows = len(languages)
    n_cols = max_domains + 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(28, 20))
    fig.suptitle('System Evals - Cross-Language Industry-wise Comparison',
                fontsize=20, fontweight='bold', y=0.98)

    if n_rows == 1:
        axes = axes.reshape(1, -1)
    if n_cols == 1:
        axes = axes.reshape(-1, 1)

    legend_handles = []
    legend_labels = []
    legend_created = False

    for row_idx, language in enumerate(languages):
        domains = language_domains[language]
        lang_data = df[df['Code Language'] == language].copy()
        print(f"🔍 Processing {language}: {len(lang_data)} records found")

        # Language label
        if n_cols > 0:
            axes[row_idx, 0].text(-0.18, 0.5, language,
                                 transform=axes[row_idx, 0].transAxes,
                                 fontsize=14, fontweight='bold',
                                 rotation=90, va='center', ha='center',
                                 color='black',
                                 bbox=dict(boxstyle='round,pad=0.5',
                                         facecolor=language_colors.get(language, 'gray'),
                                         alpha=0.7, edgecolor='white', linewidth=1))

        # Overall subplot
        overall_ax = axes[row_idx, 0]
        industry_data, industry_labels = get_overall_data_for_language(df, language)

        if industry_data and industry_labels:
            n_industries = len(industry_data)
            bar_width = 0.25
            x = np.arange(n_industries)

            for i, col in enumerate(score_cols):
                values = [data.get(col, 0.0) for data in industry_data]
                x_offset = x + (i - 1) * bar_width

                bars = overall_ax.bar(x_offset, values, width=bar_width,
                                    label=model_display_names[col],
                                    color=model_colors[col],
                                    alpha=0.8, edgecolor='white', linewidth=1)

                if not legend_created and row_idx == 0:
                    legend_handles.append(bars)
                    legend_labels.append(model_display_names[col])

            if not legend_created and row_idx == 0:
                legend_created = True

            overall_ax.set_title('Overall', fontsize=11, fontweight='bold', pad=10,
                               bbox=dict(boxstyle='round,pad=0.3',
                                       facecolor=language_colors[language], alpha=0.3))
            overall_ax.set_xticks(x)
            wrapped_labels = ["\n".join(textwrap.wrap(str(label), 10)) for label in industry_labels]
            overall_ax.set_xticklabels(wrapped_labels, rotation=0, ha='center', fontsize=8)
            overall_ax.set_xlabel('Industries', fontsize=10)
            overall_ax.set_ylabel('CodeBLEU Mean Score', fontsize=10)
            overall_ax.grid(True, axis='y', linestyle='--', alpha=0.4, color='gray')

        # Domain subplots
        for col_idx, domain in enumerate(domains):
            ax = axes[row_idx, col_idx + 1]
            domain_data = lang_data[lang_data['Domain'] == domain].copy()

            if len(domain_data) > 0:
                topics = domain_data['Subtopic'].unique().tolist()

                if topics:
                    filtered_data = []
                    for subtopic in topics:
                        subtopic_data = domain_data[domain_data['Subtopic'] == subtopic]
                        if len(subtopic_data) > 0:
                            filtered_data.append(subtopic_data.iloc[0])

                    if filtered_data:
                        plot_data = pd.DataFrame(filtered_data)
                        n_topics = len(topics)
                        bar_width = 0.25
                        x = np.arange(n_topics)

                        for i, col in enumerate(score_cols):
                            if col in plot_data.columns:
                                values = plot_data[col].values
                                x_offset = x + (i - 1) * bar_width
                                ax.bar(x_offset, values, width=bar_width,
                                      color=model_colors[col], alpha=0.8)

                        ax.set_title(domain, fontsize=11, fontweight='bold', pad=10,
                                   bbox=dict(boxstyle='round,pad=0.3',
                                           facecolor=language_colors[language], alpha=0.3))
                        ax.set_xticks(x)
                        ax.set_xticklabels([textwrap.fill(str(s), 10) for s in topics], fontsize=8)
                        ax.set_xlabel('Topics', fontsize=10)
                        ax.set_ylabel('CodeBLEU Score', fontsize=10)
                        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

        # Hide extra subplots
        for col_idx in range(len(domains) + 1, n_cols):
            axes[row_idx, col_idx].set_visible(False)

    # Legend
    if legend_handles:
        fig.legend(legend_handles, legend_labels, loc='center', bbox_to_anchor=(0.5, 0.02),
                  fontsize=12, title='Models', ncol=3, frameon=True, fancybox=True, shadow=True)

    plt.tight_layout(pad=4.0, h_pad=4.5, w_pad=2.0, rect=[0.08, 0.08, 0.95, 0.99])

    output_file = '4x4systemevals.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()  # Display in notebook
    print(f"✅ Successfully saved: {output_file}")
    return output_file

# EXECUTE: Generate 4x4 System Evaluations Cross-Language Plot
print("🔧 Generating 4x4 System Evaluations Cross-Language Plot...")

csv_file = 'RLHF data - System evals.csv'  # Update this path
output_file = create_enhanced_cross_language_domain_plot(csv_file)

if output_file:
    print(f"✅ Successfully created: {output_file}")
else:
    print("❌ Failed to create plot. Check file path and data.")

# indv. sys evals

In [ ]:
# =============================================================================
# CELL 5: INDIVIDUAL SYSTEM EVALUATIONS PLOTS
# =============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import textwrap
import seaborn as sns
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib for inline display
%matplotlib inline
plt.style.use('default')

print("✅ Imports Complete")
def create_individual_codebleu_plots(csv_file):
    """Create individual CodeBleu plots for each language-domain combination"""
    df = pd.read_csv(csv_file)

    languages = ['Python', 'JavaScript', 'Java', 'C++']
    language_domains = {
        'Python': ['Machine Learning', 'FinTech', 'EdTech'],
        'JavaScript': ['Social Networking', 'Media', 'Streaming'],
        'Java': ['E‑Commerce', 'CRM', 'Hotel'],
        'C++': ['Gaming', 'IoT', 'Security']
    }

    score_cols = [
        'Chat GPT o4 mini-high Codebleu Score',
        'Gemini 2.5 pro Codebleu Score',
        'Claude Opus 4 Codebleu Score'
    ]

    model_display_names = {
        'Chat GPT o4 mini-high Codebleu Score': 'o4-mini-high',
        'Gemini 2.5 pro Codebleu Score': 'Gemini 2.5 pro',
        'Claude Opus 4 Codebleu Score': 'Claude Opus 4'
    }

    model_colors = {
        'Chat GPT o4 mini-high Codebleu Score': '#FF8080',
        'Gemini 2.5 pro Codebleu Score': '#80FF80',
        'Claude Opus 4 Codebleu Score': '#8080FF'
    }

    language_colors = {
        "Python": "#3776ab", "JavaScript": "#f7df1e",
        "Java": "#ed8b00", "C++": "#00FFFF"
    }

    output_dir = "individual_systemevals"
    os.makedirs(output_dir, exist_ok=True)
    generated_files = []

    for language in languages:
        lang_data = df[df['Code Language'] == language].copy()

        if len(lang_data) == 0:
            print(f"Warning: No data found for {language}")
            continue

        domains = language_domains[language]

        for domain in domains:
            fig, ax = plt.subplots(figsize=(10, 10)) # Increased figure height
            domain_data = lang_data[lang_data['Domain'] == domain].copy()

            if len(domain_data) == 0:
                ax.text(0.5, 0.5, f'No data\navailable for\n{language} - {domain}',
                       ha='center', va='center', transform=ax.transAxes,
                       fontsize=14, style='italic')
                ax.set_title(f'{language} - {domain}', fontsize=16, fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.5',
                                   facecolor=language_colors.get(language, 'lightgray'),
                                   alpha=0.3), pad=20) # Adjusted title padding
                ax.axis('off')
            else:
                subtopics = domain_data['Subtopic'].tolist()
                n_subtopics = len(subtopics)
                bar_width = 0.25
                x = np.arange(n_subtopics)

                for i, col in enumerate(score_cols):
                    values = domain_data[col].values
                    x_offset = x + (i - 1) * bar_width

                    bars = ax.bar(x_offset, values, width=bar_width,
                                 label=model_display_names[col],
                                 color=model_colors[col], alpha=0.8,
                                 edgecolor='white', linewidth=1)

                ax.set_title(f'{language} - {domain}', fontsize=16, fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.5',
                                   facecolor=language_colors.get(language, 'lightgray'),
                                   alpha=0.3), pad=20) # Adjusted title padding

                ax.set_xticks(x)
                wrapped_labels = ["\n".join(textwrap.wrap(label, 15)) for label in subtopics]
                ax.set_xticklabels(wrapped_labels, rotation=0, ha='center', fontsize=10)
                ax.set_xlabel('Topics', fontsize=12, fontweight='bold')
                ax.set_ylim(0, 0.6)
                ax.set_ylabel('CodeBLEU Score', fontsize=12, fontweight='bold')
                ax.grid(True, axis='y', linestyle='--', alpha=0.4, color='gray')

                legend1 = ax.legend(loc='center', bbox_to_anchor=(0.5, -0.25),
                                  borderaxespad=0, fontsize=11, title='Models',
                                  title_fontsize=12, frameon=True, fancybox=True,
                                  shadow=True, ncol=3)


            plt.tight_layout(rect=[0, 0.08, 1, 1]) # Adjust layout to make space for title and legend

            safe_language = language.replace('+', 'plus')
            safe_domain = domain.replace(' ', '_').replace('‑', '-')
            filename = f"{safe_language}_{safe_domain}_codebleu.png"
            filepath = os.path.join(output_dir, filename)

            plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
            plt.show()  # Display in notebook
            plt.close(fig)

            generated_files.append(filepath)
            print(f"✅ Generated: {filepath}")

    return generated_files

# EXECUTE: Generate Individual System Evaluation Plots
print("🔬 Generating Individual System Evaluation Plots...")

csv_file = 'RLHF data - System evals.csv'  # Update this path

try:
    generated_files = create_individual_codebleu_plots(csv_file)

    print(f"\n🎉 Generated {len(generated_files)} individual system plots successfully!")
    print(f"📁 Output directory: individual_plots/")

    print("\n📋 Generated files:")
    for i, filepath in enumerate(generated_files, 1):
        filename = os.path.basename(filepath)
        print(f"  {i:2d}. {filename}")

except Exception as e:
    print(f"❌ Error creating individual system plots: {e}")
    import traceback
    traceback.print_exc()

# Indv. System Overall

In [ ]:
def diagnose_data_issue(csv_file):
    """Diagnose why OpenAI o4-mini-high isn't showing"""
    
    print("🔍 DIAGNOSING DATA ISSUES...")
    print("="*60)
    
    # Load the data
    df = pd.read_csv(csv_file)
    df['Language'] = df['Language'].fillna(method='ffill')
    
    print("1. AVAILABLE COLUMNS:")
    print(df.columns.tolist())
    
    print("\n2. COLUMNS CONTAINING 'o4' or 'OpenAI':")
    openai_cols = [col for col in df.columns if 'o4' in col.lower() or 'openai' in col.lower()]
    print(openai_cols)
    
    print("\n3. SAMPLE DATA:")
    print(df.head())
    
    print("\n4. DATA BY LANGUAGE:")
    for lang in ['C++', 'Python', 'Java', 'JavaScript']:
        lang_data = df[df['Language'] == lang]
        if len(lang_data) > 0:
            print(f"\n{lang}:")
            for col in openai_cols:
                if col in lang_data.columns:
                    values = lang_data[col].values
                    print(f"  {col}: {values}")
                    print(f"    - Min: {np.min(values)}, Max: {np.max(values)}")
                    print(f"    - Has nulls: {lang_data[col].isnull().any()}")
    
    return openai_cols

def create_fixed_language_industry_plots(csv_file):
    """
    Create plots with automatic column detection and debugging
    """
    
    # First diagnose the issue
    openai_cols = diagnose_data_issue(csv_file)
    
    if not openai_cols:
        print("❌ ERROR: No OpenAI o4-mini-high columns found!")
        print("Please check your CSV column names.")
        return []
    
    # Use the first available OpenAI column
    openai_col = openai_cols[0]
    print(f"\n✅ Using OpenAI column: '{openai_col}'")
    
    # Load the data
    df = pd.read_csv(csv_file)
    df['Language'] = df['Language'].fillna(method='ffill')
    
    # Define model columns - adjust based on your actual data
    score_cols = [
        openai_col,  # Use the detected OpenAI column
        'Gemini 2.5 Pro',
        'Claude Opus 4'
    ]
    
    # Check if other columns exist, if not try alternatives
    if 'Gemini 2.5 Pro' not in df.columns:
        gemini_cols = [col for col in df.columns if 'gemini' in col.lower()]
        if gemini_cols:
            score_cols[1] = gemini_cols[0]
    
    if 'Claude Opus 4' not in df.columns:
        claude_cols = [col for col in df.columns if 'claude' in col.lower()]
        if claude_cols:
            score_cols[2] = claude_cols[0]
    
    print(f"Using columns: {score_cols}")
    
    languages = ['C++', 'Python', 'Java', 'JavaScript']
    language_industries = {
        'C++': ['Gaming', 'IoT', 'Security'],
        'Python': ['Machine Learning', 'FinTech', 'EdTech'],
        'Java': ['E-Commerce', 'CRM', 'Hotel'],
        'JavaScript': ['Social Networking', 'Media', 'Streaming']
    }
    
    # Colors for models
    model_colors = {
        openai_col: '#FF8080',        # Red
        'Gemini 2.5 Pro': '#80FF80',  # Green
        'Claude Opus 4': '#8080FF'    # Blue
    }
    
    # Update colors for any alternative column names
    for col in score_cols:
        if col not in model_colors:
            if 'openai' in col.lower() or 'o4' in col.lower():
                model_colors[col] = '#FF8080'
            elif 'gemini' in col.lower():
                model_colors[col] = '#80FF80'
            elif 'claude' in col.lower():
                model_colors[col] = '#8080FF'
    
    model_display_names = {
        col: col.replace('OpenAI ', '').replace('o4-mini-high', 'o4-mini-high') 
        for col in score_cols
    }
    
    # Create output directory
    output_dir = "overall_systemevals"
    os.makedirs(output_dir, exist_ok=True)
    generated_files = []
    
    # Process each language
    for language in languages:
        print(f"\nCreating plot for {language}...")
        
        lang_data = df[df['Language'] == language].copy()
        if len(lang_data) == 0:
            continue
        
        fig, ax = plt.subplots(figsize=(10, 8))
        industries = language_industries[language]
        
        # Collect data with detailed logging
        industry_data = []
        industry_labels = []
        
        for industry in industries:
            industry_row = lang_data[lang_data['Industry'] == industry]
            if len(industry_row) > 0:
                data_point = {'industry': industry}
                
                print(f"  {industry}:")
                for col in score_cols:
                    if col in industry_row.columns:
                        value = float(industry_row[col].iloc[0])
                        data_point[col] = value
                        print(f"    {col}: {value}")
                    else:
                        data_point[col] = 0.0
                        print(f"    {col}: MISSING - using 0.0")
                
                industry_data.append(data_point)
                industry_labels.append(industry)
        
        if industry_data:
            n_industries = len(industry_data)
            bar_width = 0.25
            x = np.arange(n_industries)
            
            # Create bars with explicit debugging
            for i, model_col in enumerate(score_cols):
                values = [data.get(model_col, 0.0) for data in industry_data]
                x_offset = x + (i - 1) * bar_width
                
                print(f"    Plotting {model_col}: {values}")
                
                bars = ax.bar(x_offset, values, width=bar_width,
                             label=model_display_names.get(model_col, model_col),
                             color=model_colors.get(model_col, '#888888'),
                             alpha=0.8, edgecolor='white', linewidth=1)
                
                # Add value labels
                for j, bar in enumerate(bars):
                    height = bar.get_height()
                    if height > 0:  # Only label non-zero bars
                        ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                               f'{height:.3f}', ha='center', va='bottom', fontsize=9)
            
            # Customize plot
            ax.set_title(f'System Evals - {language} Industry-wise Comparison', 
                        fontsize=16, fontweight='bold', pad=35)
            ax.set_xticks(x)
            ax.set_xticklabels(industry_labels, rotation=0, ha='center', fontsize=12)
            ax.set_xlabel('Industry', fontsize=13, fontweight='bold')
            ax.set_ylabel('CodeBLEU Mean Score', fontsize=13, fontweight='bold')
            ax.set_ylim(0, 0.6)
            ax.grid(True, axis='y', linestyle='-', alpha=0.3, color='gray')
            ax.legend(title='Models',loc='center', bbox_to_anchor=(0.5, -0.15), ncol=3)
        
        plt.tight_layout(rect=[0, 0.1, 1, 1])
        
        safe_language = language.replace('+', 'plus')
        filename = f"{safe_language}_industry_comparison_fixed.png"
        filepath = os.path.join(output_dir, filename)
        
        plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        
        generated_files.append(filepath)
        print(f"✅ Generated: {filepath}")
    
    return generated_files

def main():
    """Main function with enhanced debugging"""
    csv_file = 'sys mean.csv'
    
    print("🚀 Starting Enhanced Language-Industry Plot Generation...")
    print("🔧 With OpenAI o4-mini-high debugging...")
    
    try:
        generated_files = create_fixed_language_industry_plots(csv_file)
        
        if generated_files:
            print(f"\n🎉 Generated {len(generated_files)} plots successfully!")
            print("📍 Check the console output above to see:")
            print("   - Which OpenAI column was used")
            print("   - Actual values for each model/industry")
            print("   - Any missing data issues")
        else:
            print("❌ No plots generated. Check the diagnostic output above.")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()